# Diabetes Risk Prediction from Health-Survey Data

**Seminar:** Advanced Applied Data Science  
**Institution:** Goethe University Frankfurt  
**Term:** Summer Semester 2026  
**Supervisor:** Prof. Dr. Kevin Bauer  
**Group:** [Group placeholder]  

**Dataset:** CDC BRFSS 2015 — Diabetes Health Indicators  
[UCI ML Repository · Dataset #891](https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators)

---

## Team & Individual Contributions

| Name | Matrikel-Nr. | Primary Contributions |
|---|---|---|
| Hasher Malik | 7632048 | NB01 Business Understanding (use-case framing, cost matrix, metric selection); NB02 Data Understanding (EDA, duplicate/label-noise analysis); NB08 Evaluation (test-set evaluation, calibration, fairness analysis, SHAP) |
| Jan Erdorf | 8748557 | NB03 Data Preparation (stratified split, feature-type taxonomy, transformation functions); NB04 Feature Diagnostics (IV/MI/KS/Spearman, VIF, feature shortlist) |
| Ilias El Ouali | 000001 | NB05 Feature Engineering (composite features, hurdle encoding, interaction terms); NB06 Modeling (14-model comparison, imbalance strategies, feature selection, top-3 shortlist) |
| Sophia X | 00002 | NB07 Model Optimization (Optuna tuning, per-model feature ablation, stacking comparison, final CatBoost selection); cross-notebook review and integration |

> **Note:** Contribution assignments reflect primary ownership. All team members participated in code review, discussion, and validation throughout the project.

## 1 · Business Understanding

### Use Case

Diabetes affects roughly 1 in 10 adults in the United States and is frequently undiagnosed. The goal of this project is to build a binary classifier that predicts whether an individual is at risk of having diabetes or prediabetes, using only self-reported health and lifestyle survey responses. Such a model could support public-health screening programs: flagging high-risk individuals for follow-up clinical testing without requiring laboratory results.

### Why Recall Matters

In a medical screening context the costs of errors are asymmetric. A **false negative** (telling a diabetic person they are low-risk) deprives them of a timely diagnosis and treatment — a high clinical cost. A **false positive** (flagging a healthy person as at-risk) leads to an unnecessary but low-cost follow-up test. This asymmetry is formalised in the project's cost matrix (NB01) and translates to a **Recall constraint: Recall ≥ 0.80**, meaning the classifier must catch at least 80 % of true positive cases. The decision threshold is therefore chosen to satisfy this constraint rather than being fixed at 0.5.

### Primary Metric: PR-AUC

The dataset is heavily imbalanced: approximately **86 % negative / 14 % positive** (no diabetes vs. prediabetes/diabetes). Under such imbalance:

- **Accuracy is rejected** as a primary metric because a trivial all-negative classifier achieves ~86 % accuracy while being clinically useless.
- **ROC-AUC** is retained as a secondary metric for context, but it is known to be optimistic under strong class imbalance because it factors in true negatives, which are abundant and easy to predict correctly.
- **PR-AUC (Area Under the Precision–Recall Curve)** is chosen as the **primary metric** because it focuses exclusively on the positive class, is directly sensitive to imbalance, and collapses to the no-skill baseline (~0.14 = class prevalence) when a model learns nothing — making it an honest benchmark. Model selection across all cross-validation experiments is based on PR-AUC.

## 2 · The Data

### Source

The dataset originates from the **Behavioral Risk Factor Surveillance System (BRFSS) 2015** survey conducted by the U.S. Centers for Disease Control and Prevention (CDC). It is published on the [UCI ML Repository as dataset #891](https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators) and can be loaded programmatically via the `ucimlrepo` package.

### Structure

| Property | Value |
|---|---|
| Samples | ~253,680 |
| Features | 21 |
| Target | `Diabetes_binary` (0 = No Diabetes, 1 = Prediabetes / Diabetes) |
| Class split | ~86.1 % negative / ~13.9 % positive |
| No-skill PR-AUC baseline | ≈ 0.14 |
| Missing values | None (verified in NB02) |

Feature types: 14 binary, 4 ordinal (GenHlth, Age, Education, Income), 2 count (MentHlth, PhysHlth), 1 continuous (BMI).

### Survey Nature and Coarse Coding

All variables are self-reported responses to a telephone survey. Several continuous quantities (age, income, education, general health) are binned into ordinal scales. BMI is reported by respondents and not clinically measured. These properties limit the precision of individual features but are representative of the information available in real-world public-health screening scenarios.

### Key Challenges

**Class Imbalance.** The ~14 % positive rate biases naive classifiers and inflates accuracy-based metrics. Addressed via `scale_pos_weight` in tree-based models (preferred over SMOTE after empirical comparison in NB06); SMOTE is available as a fallback inside `imblearn` pipelines.

**Label Noise / Positive-Unlabeled Structure.** The target records whether a respondent was *told by a doctor* that they have diabetes — it reflects diagnosis status, not disease status. Undiagnosed individuals appear as negatives despite potentially having the condition. This creates asymmetric label noise on the negative class: the true positive rate of any model is systematically underestimated. In practice, the measured PR-AUC is a **lower bound** on the model's true discriminative ability. This structure was quantified in NB02 (irreduzible label noise estimated at ~0.65 %) and is discussed further in NB08.

**Exact Duplicate Rows.** The raw dataset contains a non-trivial number of exact duplicate survey responses. Naive random splitting risks placing the same row in both train and test, inflating test performance. This is handled at the split stage in NB03 (deduplication before stratified split) and analysed in NB02.

## 3 · Methodology — CRISP-DM

The project follows the **Cross-Industry Standard Process for Data Mining (CRISP-DM)**, an iterative, phase-structured framework for applied ML projects. The six phases are:

1. **Business Understanding** — define the problem, success criteria, and metrics from a domain perspective.
2. **Data Understanding** — explore the dataset, assess quality, and identify challenges.
3. **Data Preparation** — clean, transform, and split data into train/test sets ready for modelling.
4. **Modelling** — select algorithms, tune hyperparameters, and train candidate models.
5. **Evaluation** — assess the final model against business criteria on held-out test data.
6. **Deployment** — describe how the model would be put into production (scope: this seminar addresses deployment conceptually).

In this project the preparation and modelling phases are each split across multiple notebooks to keep individual files focused.

### Notebook–Phase Mapping

| Notebook | Title | CRISP-DM Phase |
|---|---|---|
| `01_business_understanding.ipynb` | Business Understanding | Business Understanding |
| `02_data_understanding.ipynb` | Data Understanding / EDA | Data Understanding |
| `03_data_preparation.ipynb` | Data Preparation & Split | Data Preparation |
| `04_feature_diagnostics.ipynb` | Feature Diagnostics | Data Preparation |
| `05_feature_engineering.ipynb` | Feature Engineering | Data Preparation |
| `06_modeling.ipynb` | Modelling — Baseline & Comparison | Modelling |
| `07_model_optimization.ipynb` | Model Optimisation & Selection | Modelling |
| `08_evaluation.ipynb` | Evaluation, Fairness & Explainability | Evaluation |

## 4 · Reproducibility & How to Run

### Environment

Install all dependencies from the repository root:

```bash
pip install -r requirements.txt
```

Python 3.10 or later is required. Key packages include `scikit-learn`, `lightgbm`, `catboost`, `optuna`, `imbalanced-learn`, `shap`, and `ucimlrepo`.

### Random Seed

Every notebook declares `SEED = 42` at the top and passes it to all stochastic operations (train/test split, cross-validation, model initialisation, Optuna sampler). No notebook introduces additional seeds.

### Data Access

Raw data is **not** stored in the repository. NB03 fetches it automatically:

```python
from ucimlrepo import fetch_ucirepo
dataset = fetch_ucirepo(id=891)
```

NB03 then writes the stratified 80/20 split to `data/processed/` as Parquet files. All subsequent notebooks load these files. An internet connection is required for the first run.

### Run Order

Notebooks must be executed in numerical order:

```
01 → 02 → 03 → 04 → 05 → 06 → 07 → 08
```

NB03 produces the split artefacts that NB04–NB08 depend on. Running NB04 or later without first running NB03 will raise a `FileNotFoundError`.

### Repository Layout

```
diabetes-prediction-ml/
├── notebooks/          # one notebook per CRISP-DM step
├── src/
│   └── utils.py        # shared helpers (build_enriched_features, cap_bmi, …)
├── data/               # git-ignored; generated by NB03
│   ├── raw/
│   └── processed/
├── models/             # git-ignored; written by NB07
├── outputs/            # git-ignored; plots/CSVs per notebook
├── catboost_info/      # git-ignored; CatBoost training logs
├── requirements.txt
├── README.md
└── .gitignore
```

The directories `data/`, `models/`, `outputs/`, and `catboost_info/` are listed in `.gitignore` and are not tracked by Git.

In [ ]:
import sys
import platform

SEED = 42

print(f"Python  : {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"SEED    : {SEED}")
print()

_packages = [
    ("numpy",           "numpy"),
    ("pandas",          "pandas"),
    ("scikit-learn",    "sklearn"),
    ("catboost",        "catboost"),
    ("imbalanced-learn","imblearn"),
    ("ucimlrepo",       "ucimlrepo"),
]

for display, import_name in _packages:
    try:
        mod = __import__(import_name)
        version = getattr(mod, "__version__", "installed (version unknown)")
        print(f"{display:<20}: {version}")
    except ImportError:
        print(f"{display:<20}: NOT INSTALLED")